# arma_model_selection

> 对应代码：`EconometricsCode/code_in_notes/Chap.27/arma_model_selection.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.27`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.27")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们定义一个小工具程序 `display_ic`，输入 ARMA 模型的阶数 p、q、样本量 T 与对数似然值 ll，计算并打印三种信息准则：AIC、BIC 与修正的小样本 AIC（AICC）。这里参数个数以 p+q+1 计（含常数项），AIC 惩罚项为 2k，BIC 惩罚项为 $\sqrt{T}\cdot k$（这比标准 BIC 的 $\ln(T)k$ 惩罚更严），AICC 则在 AIC 基础上乘上小样本修正因子 T/(T−p−q−2)。程序以 rclass 形式把三个准则返回，便于后面在循环中比较。

In [ ]:
%%stata
cap program drop display_ic
program define display_ic, rclass
	syntax , p(integer) q(integer) t(integer) ll(real)
	tempname aic bic aicc
	local `aic'=-2*`ll'+2*(`p'+`q'+1)
	local `bic'=-2*`ll'+sqrt(`t')*(`p'+`q'+1)
	local `aicc'=-2*`ll'+2*(`p'+`q'+1)*`t'/(`t'-`p'-`q'-2)
	di "p=`p', q=`q', AIC=``aic'', BIC=``bic'', AICC=``aicc''"
	return scalar aic=``aic''
	return scalar bic=``bic''
	return scalar aicc=``aicc''
end

读入月度 PMI 数据并用 `tsset` 声明时间变量，随后画出 PMI 的时间序列图，先对序列的形态与可能的周期性做直观观察。

In [ ]:
%%stata
clear
use ../datasets/monthly_PMI.dta
tsset time

line PMI time

下面开始网格搜索 ARMA(p,q) 的阶数：p 从 0 到 5，q 从 0 到 13（跳过 p=q=0 的退化情形），对每个组合用 `arima` 估计并调用 `display_ic` 计算信息准则。我们事先用若干局部宏记录使 AICC 与 BIC 最小的 (p,q) 组合及对应最小值，循环中不断更新。之所以 q 的上界取 13，是因为 PMI 是月度数据，可能存在滞后 12 期的季节相关性，需要足够大的 MA 阶数去捕捉。

In [ ]:
%%stata
// 开始挑选
local T=_N
local minp_cc=0
local minq_cc=0
local minaicc=.
local minp_b=0
local minq_b=0
local minbic=.
forvalues p=0/5{
	forvalues q=0/13{
		if `p'==0 & `q'==0{
			continue
		}
		if `p'==0{
			qui: arima PMI, ma(1/`q')
		}
		else if `q'==0{
			qui: arima PMI, ar(1/`p')
		}
		local ll=e(ll)
		display_ic, p(`p') q(`q') t(`T') ll(`ll')
		if r(aicc)<`minaicc' {
			local minp_cc=`p'
			local minq_cc=`q'
			local minaicc=r(aicc)
		}
		if r(bic)<`minbic' {
			local minp_b=`p'
			local minq_b=`q'
			local minbic=r(bic)
		}
	}
}

循环结束后打印两个准则各自选出的最优阶数，作为后续建模的参考。

In [ ]:
%%stata
di "AICC minimized at p=`minp_cc', q=`minq_cc'"
di "BIC minimized at p=`minp_b', q=`minq_b'"

根据筛选结果，我们首先估计一个 AR(1) 模型：`arima PMI, ar(1)` 用极大似然估计带常数项的一阶自回归。估计后再次调用 `display_ic` 报告其信息准则，并用 `predict ..., residuals` 提取残差，随后画残差的时间序列图与自相关图（`ac`）。如果模型设定充分，残差应近似白噪声，自相关图上各阶自相关系数应落在置信区间内。

In [ ]:
%%stata
// 根据筛选结果，AR(1)模型
arima PMI, ar(1)
local T=_N
local p=1
local q=0
local ll=e(ll)
display_ic, p(`p') q(`q') t(`T') ll(`ll')
predict resid, residuals
line resid time
ac resid

接下来尝试在 AR(1) 基础上额外加入滞后 12 期的自回归项，即 AR 部分只包含 lag 1 和 lag 12（跳过中间各阶），以捕捉月度数据的年度季节动态。同样计算信息准则并检查残差的时间序列图与自相关图。

In [ ]:
%%stata
// 尝试其他模型
// AR(1,12)
arima PMI, ar(1 12)
local T=_N
local p=2
local q=0
local ll=e(ll)
display_ic, p(`p') q(`q') t(`T') ll(`ll')
predict resid1, residuals
line resid1 time
ac resid1

再尝试一个带季节 MA 项的设定：AR(1) 加上 12 阶的 MA 项 `ma(12)`，用移动平均项而非自回归项来刻画季节相关，比较哪种结构的残差更接近白噪声。

In [ ]:
%%stata
// ARMA
arima PMI, ar(1) ma(12)
local T=_N
local p=1
local q=1
local ll=e(ll)
display_ic, p(`p') q(`q') t(`T') ll(`ll')
predict resid2, residuals
line resid2 time
ac resid2

最后一种处理季节性的思路是直接在均值方程中加入月份虚拟变量：用 `mod(time,12)` 生成月份编号，再用 `tab ..., gen(month)` 批量生成 12 个月份哑变量，把其中 11 个作为解释变量放入 AR(1) 模型。这相当于允许每个月有不同的截距（确定性季节项），与用 ARMA 刻画随机性季节相关形成对照。注意此处把 p 设为 12 传入 `display_ic`，只是为了让信息准则的惩罚项大致反映 11 个月份虚拟变量加一个 AR 系数的模型复杂度。

In [ ]:
%%stata
// 加入月份虚拟变量
gen pmonth=mod(time, 12)
tab pmonth, gen(month)
arima PMI month1-month11, ar(1)
local T=_N
local p=12
local q=0
local ll=e(ll)
display_ic, p(`p') q(`q') t(`T') ll(`ll')
predict resid3, residuals
line resid3 time
ac resid3